In [1]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_id = "bert-base-uncased"

/home/dell/Documents/agent_planner/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
label2id = {"add": 0, "delete": 1, "get_event": 2,"time_block":3}
id2label = {v: k for k, v in label2id.items()}

In [3]:
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSequenceClassification.from_pretrained(model_id,num_labels=4,id2label=id2label,label2id=label2id)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 12278.66it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpo

In [4]:
for name,param in model.named_parameters():
    if name.startswith("classifier"):
        param.require_grad = True
    else:
        param.require_grad = False

In [5]:
def format_data(data):
    data["label"] = [label2id[label] for label in data["label"]]
    tokenized = tokenizer(data["prompt"],truncation=True)
    data["token_type_ids"] = tokenized["token_type_ids"]
    data["input_ids"] = tokenized["input_ids"]
    data["attention_mask"] = tokenized["attention_mask"]
    return data

In [6]:
from datasets import Dataset

data = Dataset.from_json("training_data.json").shuffle(seed=42)
eval_data = Dataset.from_json("eval_data.json")

In [7]:
import json


# -----------------------------

def load_prompts(data):
    prompts = []
    prompts.extend(item["prompt"] for item in data)
    return prompts

train_prompts = load_prompts(data)
eval_prompts = load_prompts(eval_data)

train_set = set(train_prompts)
eval_set = set(eval_prompts)

# 1. Exact duplicate leakage (eval prompt appears verbatim in train)
leaked = eval_set & train_set

# 2. Duplicates within train itself
train_dupes = len(train_prompts) - len(train_set)

# 3. Duplicates within eval itself
eval_dupes = len(eval_prompts) - len(eval_set)

# 4. Near-duplicate check (case/whitespace-insensitive)
def normalize(s):
    return " ".join(s.lower().split())

train_norm = set(normalize(p) for p in train_prompts)
eval_norm = set(normalize(p) for p in eval_prompts)
near_leaked = eval_norm & train_norm

print(f"Train size: {len(train_prompts)} (unique: {len(train_set)}, internal dupes: {train_dupes})")
print(f"Eval size:  {len(eval_prompts)} (unique: {len(eval_set)}, internal dupes: {eval_dupes})")
print()
print(f"Exact leakage (eval prompts found verbatim in train): {len(leaked)}")
if leaked:
    for p in list(leaked)[:10]:
        print("  -", p)

print()
print(f"Near-duplicate leakage (case/whitespace-insensitive): {len(near_leaked)}")
if near_leaked:
    for p in list(near_leaked)[:10]:
        print("  -", p)

if not leaked and not near_leaked:
    print("\n✅ No data leakage detected between train and eval.")
else:
    print("\n❌ Leakage detected — clean it before training.")

Train size: 800 (unique: 799, internal dupes: 1)
Eval size:  120 (unique: 120, internal dupes: 0)

Exact leakage (eval prompts found verbatim in train): 0

Near-duplicate leakage (case/whitespace-insensitive): 0

✅ No data leakage detected between train and eval.


In [8]:
from transformers import DataCollatorWithPadding
data = data.map(format_data,batched=True)
eval_data = eval_data.map(format_data,batched=True)
data_collat = DataCollatorWithPadding(tokenizer=tokenizer)

In [9]:
import numpy as np
import evaluate

accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    
    accuracy = accuracy_metric.compute(predictions=predictions, references=labels)
    f1 = f1_metric.compute(predictions=predictions, references=labels, average="weighted")
    
    return {
        "accuracy": accuracy["accuracy"],
        "f1": f1["f1"]
    }

In [10]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./bert_output",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    num_train_epochs=3,
    save_strategy="epoch"
)

trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collat,
    train_dataset=data,
    eval_dataset=eval_data,
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)

In [11]:
trainer.evaluate()

Training Loss,Validation Loss,Step,Accuracy,F1
No log,1.417642,0,0.250000,0.101351


{'eval_loss': 1.4176424741744995,
 'eval_accuracy': 0.25,
 'eval_f1': 0.10135135135135136}

In [12]:
trainer.train()

Step,Training Loss


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.53it/s]


TrainOutput(global_step=150, training_loss=0.32511853535970053, metrics={'train_runtime': 21.0335, 'train_samples_per_second': 114.104, 'train_steps_per_second': 7.131, 'total_flos': 26747363520384.0, 'train_loss': 0.32511853535970053, 'epoch': 3.0})

In [13]:
trainer.evaluate()

Training Loss,Validation Loss,Step,Accuracy,F1
No log,0.018844,150,1.000000,1.000000


{'eval_loss': 0.01884414814412594, 'eval_accuracy': 1.0, 'eval_f1': 1.0}

In [14]:
import torch
def classify(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding="max_length", max_length=64).to("cuda:0")
    
    with torch.no_grad():
        outputs = trainer.model(**inputs)
    
    probs = torch.softmax(outputs.logits, dim=-1)[0]
    predicted_id = torch.argmax(probs).item()
    
    return {
        "intent": id2label[predicted_id],
        "confidence": round(probs[predicted_id].item(), 4),
        "all_scores": {id2label[i]: round(p.item(), 4) for i, p in enumerate(probs)}
    }

classify("hehe remove yesterday events")
# → {"intent": "reminder", "confidence": 0.9821, "all_scores": {...}}

{'intent': 'delete',
 'confidence': 0.9368,
 'all_scores': {'add': 0.0103,
  'delete': 0.9368,
  'get_event': 0.0433,
  'time_block': 0.0096}}

In [15]:
trainer.model.save_pretrained("bert-schduler-classification")

Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.13s/it]


In [16]:
trainer.model.push_to_hub("bert-AIPlanner-classification")

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.41it/s]
Processing Files (1 / 1): 100%|██████████|  438MB /  438MB, 10.5MB/s  
New Data Upload: 100%|██████████|  431MB /  431MB, 10.5MB/s  


CommitInfo(commit_url='https://huggingface.co/sea-rod/bert-AIPlanner-classification/commit/c7c4d74e88ed06e38638555a594964eca63af156', commit_message='Upload BertForSequenceClassification', commit_description='', oid='c7c4d74e88ed06e38638555a594964eca63af156', pr_url=None, repo_url=RepoUrl('https://huggingface.co/sea-rod/bert-AIPlanner-classification', endpoint='https://huggingface.co', repo_type='model', repo_id='sea-rod/bert-AIPlanner-classification'), pr_revision=None, pr_num=None)

In [17]:
trainer.processing_class.push_to_hub("bert-AIPlanner-classification")

CommitInfo(commit_url='https://huggingface.co/sea-rod/bert-AIPlanner-classification/commit/f3c36a5abc628845c3cbe9c19056d0c62c75c945', commit_message='Upload tokenizer', commit_description='', oid='f3c36a5abc628845c3cbe9c19056d0c62c75c945', pr_url=None, repo_url=RepoUrl('https://huggingface.co/sea-rod/bert-AIPlanner-classification', endpoint='https://huggingface.co', repo_type='model', repo_id='sea-rod/bert-AIPlanner-classification'), pr_revision=None, pr_num=None)

In [18]:
from transformers import pipeline

pipe = pipeline("text-classification",model=trainer.model,tokenizer=trainer.processing_class)


In [19]:
pipe("hello")

[{'label': 'get_event', 'score': 0.767839252948761}]